In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import os 
import tqdm
import glob
import tensorflow 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data.experimental import AUTOTUNE
from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import RandomRotation, RandomZoom
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import applications
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from keras import layers

from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow
from skimage.transform import resize

from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import Precision, AUC,Recall
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation, MaxPool2D, Conv2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.densenet import DenseNet169
import copy
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import cv2
import keras
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from keras.applications.vgg16 import VGG16,preprocess_input
from keras.applications.vgg19 import VGG19,preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
import tensorflow_addons as tfa
from tensorflow_addons.optimizers import RectifiedAdam
from tensorflow_addons.optimizers import AdamW
from PIL import Image

labels = pd.read_csv('AlzheimersFLAIR/patient_labels.csv')
labels

In [ ]:
import sys

import pandas as pd
import sklearn as sk
import tensorflow as tf
import keras
import platform

print(f"Python Platform: {platform.platform()}")
print(f"Tensor Flow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print()
print(f"Python {sys.version}")
print(f"Pandas {pd.__version__}")
print(f"Scikit-Learn {sk.__version__}")
gpu = len(tf.config.list_physical_devices('GPU'))>0
print("GPU is", "available" if gpu else "NOT AVAILABLE")

In [ ]:
from PIL import Image

def load_and_preprocess_images(file_path):
    image = np.array(Image.open(file_path))
    return image

In [ ]:
import nibabel as nib

x_train = []
y_train = []
x_val = []
y_val = []
x_test = []
y_test = []
directory = 'Splitted2D-21'

x_train = []
y_train = []
x_val = []
y_val = []
x_test = []
y_test = []

used_patients_train = []
used_patients_val = []
used_patients_test = []


for subset in os.listdir(directory):
    if (subset != ".DS_Store"):
        print(subset)
        subset_path = os.path.join(directory, subset)
        for diagnosis in os.listdir(subset_path):
            if (diagnosis != ".DS_Store"):
                diagnosis_path = os.path.join(subset_path, diagnosis)
                for filename in os.listdir(diagnosis_path):
                    print(diagnosis)
                    if (filename != ".DS_Store"):
                        file_path = os.path.join(diagnosis_path, filename)

                        if (subset == 'train'):
                            x_train.append(load_and_preprocess_images(file_path))

                            y_train.append(int(diagnosis))
                            
                            used_patients_train.append(filename)

                        elif (subset == 'val'):
                            x_val.append(load_and_preprocess_images(file_path))

                            y_val.append(int(diagnosis))
                            
                            used_patients_val.append(filename)
                        else:
                            x_test.append(load_and_preprocess_images(file_path))

                            y_test.append(int(diagnosis))
                            
                            used_patients_test.append(filename)


In [ ]:
plt.figure()
plt.imshow(x_train[0])
plt.show()

In [ ]:
#tf.config.run_functions_eagerly(False)

class CustomDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, directory, batch_size, weighted = True):
        self.volume_tensor, self.label_list = self.load_and_preprocess_images(directory)
        self.batch_size = batch_size
        self.weighted = weighted
    
    def __len__(self):
        return int(np.ceil(len(self.volume_tensor) / self.batch_size))
    
    def __getitem__(self, index):
        batch_indices = []
        if self.weighted:
            batch_indices = self.get_weighted_random_sample()
        
        else:
            batch_indices = np.random.shuffle(np.arange(len(self.label_list)))
        batch_indices = np.random.choice(np.arange(len(self.label_list)), size = batch_size, replace = False)
        #batch_x = tf.gather(self.volume_tensor, batch_indices)
        batch_x = np.stack([self.volume_tensor[i] for i in batch_indices])
        
        batch_y = [self.label_list[i] for i in batch_indices]
        
        out_y = []
        for i in batch_y:
            label = [0, 0, 0, 0]
            label[i] += 1
            out_y.append(np.array(label))
        
        return batch_x, np.array(out_y)
    
    def load_and_preprocess_images(self, directory):
        # load images
        label_list = []
        image_list = []
        for label in os.listdir(directory):
            #print(label)
            if (label != ".DS_Store"):
                label_path = os.path.join(directory, label)
                
                for img in os.listdir(label_path):
                    if (img.endswith('.jpg')):
                        img_path = os.path.join(label_path, img)
                        
                        label_list.append(int(label))
                        image_list.append(np.expand_dims(np.array(Image.open(img_path)), axis = -1))
        image_tensor = tf.image.grayscale_to_rgb(tf.convert_to_tensor(image_list))
        #rgb_list = [np.stack((img,) * 3, axis=-1) for img in image_list]
        
        data_augmentor = Sequential([
            layers.Resizing(224, 224),
            layers.Rescaling(1./255),
            layers.RandomRotation(0.1, fill_mode = 'constant'),
            #layers.RandomZoom((.5, .5)),
            # randomzoom without using 'constant' interpolates with corners of the brain
            # using 'constant' makes randomzoom not actually zoom
        ])

#        augmented_list = []
#        for i in rgb_list:
#            augmented_list.append(data_augmentor(i))

        augmented_list = []
        for i in image_tensor:
            augmented_list.append(data_augmentor(i).numpy())
        print(type(augmented_list[0]))
        tensors = np.stack(augmented_list)
        #print(np.shape(tensors)

#         plt.figure()
#         plt.imshow(augmented_list[0])
#         plt.show()

        return tensors, label_list
        
    
    def get_weighted_random_sample(self):
        class_frequencies = np.bincount(self.label_list)

        # Calculate the inverse of class frequencies as probabilities
        class_probabilities = 1.0 / class_frequencies
        #print(class_probabilities)
        # Normalize the probabilities to sum to 1
        class_probabilities /= np.sum(class_probabilities)
        #print(self.label_list)
        index_probabilities = [class_probabilities[i] for i in self.label_list]
        
        index_probabilities /= np.sum(index_probabilities)
        
        # Randomly choose sample indices with replacement using class probabilities
        random_indices = np.random.choice(np.arange(len(self.label_list)), size=self.batch_size, 
                                          replace=False, p=index_probabilities)
        return random_indices

In [ ]:
batch_size = 32
train_dir = 'Splitted2D-19/train'
val_dir = 'Splitted2D-19/val'
test_dir = 'Splitted2D-19/test'

train_generator = CustomDataGenerator(directory = train_dir, batch_size = batch_size)
val_generator = CustomDataGenerator(directory = val_dir, batch_size = batch_size, weighted = False)
test_generator = CustomDataGenerator(directory = test_dir, batch_size = batch_size, weighted = False)

In [ ]:
for x, y in test_generator:
    print(y)

In [ ]:
type(train_generator)

In [ ]:
freq_total = [0, 0, 0, 0]
for x, y in train_generator:
    
    
    
    print(np.shape(x))
    plt.figure()
    plt.imshow(x[0], cmap = 'gray')
    plt.show()

#     outputs = []
#     for i in y:
#         outputs.append(np.argmax(i))
#     batch_freq = np.bincount(outputs)
#     print(batch_freq)
#     freq_total += batch_freq
    #print(y)

print(freq_total)

In [ ]:
#from keras.optimizers.legacy import SGD
input_shape = (224,224, 3)

#Create an instance of the VGG16 model
vgg16 = VGG16(include_top=False, input_shape=input_shape,
                   weights='imagenet')

# Freeze the layers of the VGG16 model
for layer in vgg16.layers:
    layer.trainable = False
    
# Create a new model with additional layers
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()

prediction_layer = keras.layers.Dense(4,activation='softmax')

model = tf.keras.Sequential([vgg16, global_average_layer,
    keras.layers.BatchNormalization(),  
    keras.layers.Dropout(0.25),
    keras.layers.Dense(2048, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(512, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(256, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(64, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.25),
    prediction_layer
])

model.compile(optimizer=tf.keras.optimizers.legacy.SGD(learning_rate = 0.02, decay = 0.0001), 
              loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:


history=model.fit(train_generator,
                        validation_data=val_generator,
                        steps_per_epoch=len(train_generator),
                        epochs = 75,
                        verbose = 1)

In [ ]:
result = model.evaluate(train_generator)
train_loss = result[0]
train_accuracy = result[1]
train_AUC = result[2]
train_pre = result[3]
train_rec = result[4]
print(f'Train Loss = {train_loss}')
print(f'Train Accuracy = {train_accuracy}')
print(f'Train AUC = {train_AUC}')
print(f'Train Precision = {train_pre}')
print(f'Train Recall = {train_rec}')

result = model.evaluate(test_generator)
test_loss = result[0]
test_accuracy = result[1]
test_AUC = result[2]
test_pre = result[3]
test_rec = result[4]
print(f'Test Loss = {test_loss}')
print(f'Test Accuracy = {test_accuracy}')
print(f'Test AUC = {test_AUC}')
print(f'Test Precision = {test_pre}')
print(f'Test Recall = {test_rec}')

In [ ]:
y_pred=[]
y_true = []
for images, label in test_generator:
    for l in label:
        y_true.append(np.argmax(l))
    
    Y_pred = [np.argmax(l) for l in model.predict(images)]
    for l in Y_pred:
        y_pred.append(l)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, classes, normalize=False, cmap=plt.cm.Blues):
    if normalize:
#         cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = "Normalized Confusion Matrix"
    else:
        title = "Confusion Matrix"

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90)
    plt.yticks(tick_marks, classes)

    fmt = 'd'
#     '.2f' if normalize else 
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

cm = confusion_matrix(y_true[:348], y_pred)

class_names = [0, 1, 2, 3]

plt.figure(figsize=(8, 6))
plot_confusion_matrix(cm, class_names, normalize=True)
plt.show()